In [1]:
# We will use the model trained on the breast cancer dataset and then make the predictions on the given protein and smile sequences and the predictions will be 0/1 for the interaction if possible.

In [10]:
import torch

In [11]:
file_path = "../output/model/b_cancer--radius2--ngram3--dim10--layer_gnn3--window11--layer_cnn3--layer_output3--lr1e-3--lr_decay0.5--decay_interval10--weight_decay1e-6--iteration100"

model = torch.load(f=file_path, map_location="cpu")

In [14]:
import torch
import pickle
import numpy as np
from rdkit import Chem
from collections import defaultdict
import torch.nn as nn

In [26]:
# ========= LOAD MODEL AND DICTIONARIES ========= #

# Change this to your saved model checkpoint
MODEL_PATH = "../output/model/b_cancer--radius2--ngram3--dim10--layer_gnn3--window11--layer_cnn3--layer_output3--lr1e-3--lr_decay0.5--decay_interval10--weight_decay1e-6--iteration100"

# Same values used during training
radius = 2
ngram = 3

dim=10
layer_gnn=3
side=5
window=2 * side + 1
layer_cnn=3
layer_output=3
lr=1e-3
lr_decay=0.5
decay_interval=10
weight_decay=1e-6
iteration=100

# Load pickled dictionaries
dir_input = "../dataset/b_cancer/input/radius2_ngram3/"
with open(dir_input + "fingerprint_dict.pickle", "rb") as f:
    fingerprint_dict = pickle.load(f)
with open(dir_input + "word_dict.pickle", "rb") as f:
    word_dict = pickle.load(f)

n_fingerprint = len(fingerprint_dict)
n_word = len(word_dict)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class CompoundProteinInteractionPrediction(nn.Module):
    def __init__(self):
        super(CompoundProteinInteractionPrediction, self).__init__()
        self.embed_fingerprint = nn.Embedding(n_fingerprint, dim)
        self.embed_word = nn.Embedding(n_word, dim)
        self.W_gnn = nn.ModuleList([nn.Linear(dim, dim)
                                    for _ in range(layer_gnn)])
        self.W_cnn = nn.ModuleList([nn.Conv2d(
                     in_channels=1, out_channels=1, kernel_size=2*window+1,
                     stride=1, padding=window) for _ in range(layer_cnn)])
        self.W_attention = nn.Linear(dim, dim)
        self.W_out = nn.ModuleList([nn.Linear(2*dim, 2*dim)
                                    for _ in range(layer_output)])
        self.W_interaction = nn.Linear(2*dim, 2)

    def gnn(self, xs, A, layer):
        for i in range(layer):
            hs = torch.relu(self.W_gnn[i](xs))
            xs = xs + torch.matmul(A, hs)
        # return torch.unsqueeze(torch.sum(xs, 0), 0)
        return torch.unsqueeze(torch.mean(xs, 0), 0)

    def attention_cnn(self, x, xs, layer):
        """The attention mechanism is applied to the last layer of CNN."""

        xs = torch.unsqueeze(torch.unsqueeze(xs, 0), 0)
        for i in range(layer):
            xs = torch.relu(self.W_cnn[i](xs))
        xs = torch.squeeze(torch.squeeze(xs, 0), 0)

        h = torch.relu(self.W_attention(x))
        hs = torch.relu(self.W_attention(xs))
        weights = torch.tanh(F.linear(h, hs))
        ys = torch.t(weights) * hs

        # return torch.unsqueeze(torch.sum(ys, 0), 0)
        return torch.unsqueeze(torch.mean(ys, 0), 0)

    def forward(self, inputs):

        fingerprints, adjacency, words = inputs

        """Compound vector with GNN."""
        fingerprint_vectors = self.embed_fingerprint(fingerprints)
        compound_vector = self.gnn(fingerprint_vectors, adjacency, layer_gnn)

        """Protein vector with attention-CNN."""
        word_vectors = self.embed_word(words)
        protein_vector = self.attention_cnn(compound_vector,
                                            word_vectors, layer_cnn)

        """Concatenate the above two vectors and output the interaction."""
        cat_vector = torch.cat((compound_vector, protein_vector), 1)
        for j in range(layer_output):
            cat_vector = torch.relu(self.W_out[j](cat_vector))
        interaction = self.W_interaction(cat_vector)

        return interaction

    def __call__(self, data, train=True):

        inputs, correct_interaction = data[:-1], data[-1]
        predicted_interaction = self.forward(inputs)

        if train:
            loss = F.cross_entropy(predicted_interaction, correct_interaction)
            return loss
        else:
            correct_labels = correct_interaction.to('cpu').data.numpy()
            ys = F.softmax(predicted_interaction, 1).to('cpu').data.numpy()
            predicted_labels = list(map(lambda x: np.argmax(x), ys))
            predicted_scores = list(map(lambda x: x[1], ys))
            return correct_labels, predicted_labels, predicted_scores

model = CompoundProteinInteractionPrediction().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

# ========== PREPROCESS FUNCTIONS (EXACTLY FROM preprocessing.py) ========== #

def create_atoms(mol):
    atoms = [a.GetSymbol() for a in mol.GetAtoms()]
    for a in mol.GetAromaticAtoms():
        i = a.GetIdx()
        atoms[i] = (atoms[i], 'aromatic')
    atoms = [fingerprint_dict[a] for a in atoms]
    return np.array(atoms)

def create_ijbonddict(mol):
    from collections import defaultdict
    i_jbond_dict = defaultdict(lambda: [])
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bond = str(b.GetBondType())
        i_jbond_dict[i].append((j, bond))
        i_jbond_dict[j].append((i, bond))
    return i_jbond_dict

def extract_fingerprints(atoms, i_jbond_dict, radius):
    nodes = atoms.copy()
    for _ in range(radius):
        new_nodes = []
        for i, neighbors in i_jbond_dict.items():
            neigh = [(nodes[j], edge) for j, edge in neighbors]
            fp = (nodes[i], tuple(sorted(neigh)))
            if fp not in fingerprint_dict:
                fingerprint_dict[fp] = len(fingerprint_dict)
            new_nodes.append(fingerprint_dict[fp])
        nodes = np.array(new_nodes)
    return nodes

def create_adjacency(mol):
    return np.array(Chem.GetAdjacencyMatrix(mol))

def split_sequence(seq, n):
    seq = '-' + seq + '='
    return np.array([word_dict[seq[i:i+n]] for i in range(len(seq)-n+1)])

# ========== PREDICT FUNCTION ========== #

def predict(smiles, sequence):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    atoms = create_atoms(mol)
    i_jbond = create_ijbonddict(mol)
    fingerprints = extract_fingerprints(atoms, i_jbond, radius)
    adjacency = create_adjacency(mol)
    words = split_sequence(sequence, ngram)

    # Convert to torch tensors
    fingerprints = torch.LongTensor(fingerprints).to(device)
    adjacency = torch.FloatTensor(adjacency).to(device)
    words = torch.LongTensor(words).to(device)

    inputs = (fingerprints, adjacency, words)

    with torch.no_grad():
        logits = model.forward(inputs)
        prob = torch.softmax(logits, dim=1)[0][1].item()

    return float(prob)  # probability of interaction


# ========== EXAMPLE USAGE ========== #

if __name__ == "__main__":
    test_smiles = "CCO"  # replace
    test_protein = "MKVLWAALLVTFLAGCQAKVE"  # replace

    score = predict(test_smiles, test_protein)
    print(f"Predicted interaction probability = {score:.4f}")


KeyError: 'C'

In [23]:
import os
import sys
import pickle
import numpy as np
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem

# ---------------------------
# Configuration / arguments
# ---------------------------
# Usage:
# python predict_single.py <DATASET> <radius> <ngram> <model_path> "<SMILES>" "<PROTEIN_SEQUENCE>"
#
# Example:
# python predict_single.py b_cancer 2 3 ../output/model/b_cancer--radius2--ngram3--dim10--layer_gnn3--window5--layer_cnn3--layer_output3--lr0.001--lr_decay0.5--decay_interval10--weight_decay1e-06--iteration100 "CC(=O)Oc1ccccc1C(=O)O" "MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYR"

# if len(sys.argv) != 7:
#     print("Usage: python predict_single.py <DATASET> <radius> <ngram> <model_path> \"<SMILES>\" \"<PROTEIN_SEQUENCE>\"")
#     sys.exit(1)



MODEL_PATH = "../output/model/b_cancer--radius2--ngram3--dim10--layer_gnn3--window11--layer_cnn3--layer_output3--lr1e-3--lr_decay0.5--decay_interval10--weight_decay1e-6--iteration100"

# Same values used during training
DATASET="b_cancer"
radius = 2
ngram = 3

dim=10
layer_gnn=3
side=5
window=2 * side + 1
layer_cnn=3
layer_output=3
lr=1e-3
lr_decay=0.5
decay_interval=10
weight_decay=1e-6
iteration=100

# DATASET, radius, ngram, model_path, SMILES, SEQUENCE = sys.argv[1:]
radius = int(radius)
ngram = int(ngram)

# ---------------------------
# Helper: load dicts & model meta
# ---------------------------
dir_input = ('../dataset/' + DATASET + '/input/'
             'radius' + str(radius) + '_ngram' + str(ngram) + '/')

fingerprint_dict_path = os.path.join(dir_input, 'fingerprint_dict.pickle')
word_dict_path = os.path.join(dir_input, 'word_dict.pickle')
smiles_file = os.path.join(dir_input, 'Smiles.txt')

if not os.path.exists(fingerprint_dict_path) or not os.path.exists(word_dict_path):
    raise FileNotFoundError(f"Could not find fingerprint_dict or word_dict in {dir_input}")

with open(fingerprint_dict_path, 'rb') as f:
    fingerprint_dict = pickle.load(f)

with open(word_dict_path, 'rb') as f:
    word_dict = pickle.load(f)

# We need n_fingerprint and n_word for model embedding sizes
n_fingerprint = len(fingerprint_dict)
n_word = len(word_dict)

print(f"Loaded fingerprint_dict (size={n_fingerprint}) and word_dict (size={n_word}) from {dir_input}")
print("WARNING: prediction will be reliable only if fingerprint_dict keys are buildable from atom symbols/tuples.")
# ---------------------------
# Re-implement minimal parts of preprocessing
# ---------------------------

def create_atoms_for_prediction(mol):
    """Return list of atom identity tokens (strings or tuples) for fallback matching with fingerprint_dict keys.
    We DO NOT rely on the original atom_dict numeric mapping (which may be absent)."""
    atoms = [a.GetSymbol() for a in mol.GetAtoms()]
    for a in mol.GetAromaticAtoms():
        i = a.GetIdx()
        atoms[i] = (atoms[i], 'aromatic')
    return atoms  # list of strings or tuples

def create_ijbonddict(mol):
    i_jbond = defaultdict(list)
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bond = str(b.GetBondType())  # use bond type string as token
        i_jbond[i].append((j, bond))
        i_jbond[j].append((i, bond))
    return i_jbond

def extract_fingerprints_for_prediction(atoms_tokens, i_jbond_dict, radius):
    """
    Try to create fingerprint keys compatible with the saved fingerprint_dict.
    For base atoms, we attempt to find a key in fingerprint_dict directly using the atom token,
    otherwise fallback to 0 (unknown).
    For higher radii, we build tuple structures similar to the original code and lookup them in fingerprint_dict.
    If a built key isn't present in fingerprint_dict, we fallback to 0.
    """
    # Base case
    if (len(atoms_tokens) == 1) or (radius == 0):
        fps = []
        for a in atoms_tokens:
            # try atom token as-is
            if a in fingerprint_dict:
                fps.append(fingerprint_dict[a])
            else:
                # try more variants: symbol string if tuple passed
                if isinstance(a, tuple) and a in fingerprint_dict:
                    fps.append(fingerprint_dict[a])
                else:
                    fps.append(0)  # unknown-index fallback
        return np.array(fps, dtype=np.int64)

    # General WL-like expansion
    nodes = list(atoms_tokens)  # tokens (strings/tuples or integers)
    i_jedge_dict = i_jbond_dict.copy()

    # We'll represent node labels as tokens (not numeric id)
    for _ in range(radius):
        fingerprints = []
        for i, j_edge in i_jedge_dict.items():
            neighbors = []
            for j, edge in j_edge:
                neighbors.append((nodes[j], edge))
            fingerprint_key = (nodes[i], tuple(sorted(neighbors)))
            if fingerprint_key in fingerprint_dict:
                fingerprints.append(fingerprint_dict[fingerprint_key])
            else:
                # try to convert nodes[i] to string if it's numeric-like etc.
                # fallback to 0
                fingerprints.append(0)
        # update nodes
        nodes = fingerprints

        # update edges: map using both_side tuple tokens - attempt to match keys in some sensible way
        _i_jedge_dict = defaultdict(list)
        for i, j_edge in i_jedge_dict.items():
            for j, edge in j_edge:
                # both_side uses updated nodes values (integers)
                both_side = tuple(sorted((nodes[i], nodes[j])))
                edge_key = (both_side, edge)
                # The original code used edge_dict[(both_side, edge)] to produce integer edge tokens which were then used.
                # We don't have edge_dict, so we'll just reuse `edge` string token as a fallback.
                _i_jedge_dict[i].append((j, edge))
        i_jedge_dict = _i_jedge_dict

    return np.array(fingerprints, dtype=np.int64)

def create_adjacency_for_prediction(mol):
    adj = Chem.GetAdjacencyMatrix(mol)
    return np.array(adj, dtype=np.float32)

def split_sequence_for_prediction(sequence, ngram):
    # build augmented sequence same way original code did: '-' + sequence + '='
    s = '-' + sequence + '='
    words = []
    for i in range(len(s) - ngram + 1):
        token = s[i:i+ngram]
        if token in word_dict:
            words.append(word_dict[token])
        else:
            # fallback to 0 for unknown tokens
            words.append(0)
    return np.array(words, dtype=np.int64)

# ---------------------------
# Build model class (must match training)
# ---------------------------
# We replicate the architecture defined in run_training.py and instantiate with correct sizes.
class CompoundProteinInteractionPrediction(nn.Module):
    def __init__(self, n_fingerprint, n_word, dim, layer_gnn, window, layer_cnn, layer_output):
        super(CompoundProteinInteractionPrediction, self).__init__()
        self.embed_fingerprint = nn.Embedding(n_fingerprint if n_fingerprint>0 else 1, dim)
        self.embed_word = nn.Embedding(n_word if n_word>0 else 1, dim)
        self.W_gnn = nn.ModuleList([nn.Linear(dim, dim)
                                    for _ in range(layer_gnn)])
        self.W_cnn = nn.ModuleList([nn.Conv2d(
                     in_channels=1, out_channels=1, kernel_size=2*window+1,
                     stride=1, padding=window) for _ in range(layer_cnn)])
        self.W_attention = nn.Linear(dim, dim)
        self.W_out = nn.ModuleList([nn.Linear(2*dim, 2*dim)
                                    for _ in range(layer_output)])
        self.W_interaction = nn.Linear(2*dim, 2)

    def gnn(self, xs, A, layer):
        for i in range(layer):
            hs = torch.relu(self.W_gnn[i](xs))
            xs = xs + torch.matmul(A, hs)
        return torch.unsqueeze(torch.mean(xs, 0), 0)

    def attention_cnn(self, x, xs, layer):
        xs = torch.unsqueeze(torch.unsqueeze(xs, 0), 0)
        for i in range(layer):
            xs = torch.relu(self.W_cnn[i](xs))
        xs = torch.squeeze(torch.squeeze(xs, 0), 0)

        h = torch.relu(self.W_attention(x))
        hs = torch.relu(self.W_attention(xs))
        # weights shape mismatch guard: if dims mismatch, we'll fallback to average
        try:
            weights = torch.tanh(F.linear(h, hs))
            ys = torch.t(weights) * hs
            return torch.unsqueeze(torch.mean(ys, 0), 0)
        except Exception:
            return torch.unsqueeze(torch.mean(hs, 0), 0)

    def forward(self, inputs):
        fingerprints, adjacency, words = inputs

        fingerprint_vectors = self.embed_fingerprint(fingerprints)
        compound_vector = self.gnn(fingerprint_vectors, adjacency, layer_gnn)

        word_vectors = self.embed_word(words)
        protein_vector = self.attention_cnn(compound_vector,
                                            word_vectors, layer_cnn)

        cat_vector = torch.cat((compound_vector, protein_vector), 1)
        for j in range(layer_output):
            cat_vector = torch.relu(self.W_out[j](cat_vector))
        interaction = self.W_interaction(cat_vector)

        return interaction

    def predict(self, inputs):
        self.eval()
        with torch.no_grad():
            predicted = self.forward(inputs)
            probs = F.softmax(predicted, dim=1).cpu().numpy()[0]
            label = int(np.argmax(probs))
            score = float(probs[1]) if len(probs) > 1 else float(probs[0])
        return label, score, probs

# ---------------------------
# Load model state and hyperparameters
# ---------------------------
# The training `save_model` call saved `torch.save(model.state_dict(), filename)`.
# We need to know dim, layer_gnn, window, layer_cnn, layer_output used when training.
# If you used the same batch script included, default values are shown earlier.
# If your model file is accompanied by a small metadata file (recommended), load it.
#
# Fallbacks (common defaults from your batch file):
dim = 10
layer_gnn = 3
window = 5  # side=5 => window = 2*side+1 earlier; but script passed window variable already; using value from your batch script: side=5 => window = 11? 
# NOTE: In your provided batch file you set side=5 and then window = 2*%side% + 1 => window=11.
# But in run_training you already pass `window` as an integer. To be safe we'll try the environment variable file if present.
layer_cnn = 3
layer_output = 3

# Try to infer hyperparameters from model filename (best-effort)
fname = os.path.basename(MODEL_PATH)
if 'dim' in fname:
    try:
        # attempt to parse "dim10" pattern
        import re
        m = re.search(r"dim(\d+)", fname)
        if m:
            dim = int(m.group(1))
    except Exception:
        pass
if 'layer_gnn' in fname:
    try:
        import re
        m = re.search(r"layer_gnn(\d+)", fname)
        if m:
            layer_gnn = int(m.group(1))
    except Exception:
        pass
if 'window' in fname:
    try:
        import re
        m = re.search(r"window(\d+)", fname)
        if m:
            window = int(m.group(1))
    except Exception:
        pass
if 'layer_cnn' in fname:
    try:
        import re
        m = re.search(r"layer_cnn(\d+)", fname)
        if m:
            layer_cnn = int(m.group(1))
    except Exception:
        pass
if 'layer_output' in fname:
    try:
        import re
        m = re.search(r"layer_output(\d+)", fname)
        if m:
            layer_output = int(m.group(1))
    except Exception:
        pass

print(f"Using hyperparams dim={dim}, layer_gnn={layer_gnn}, window={window}, layer_cnn={layer_cnn}, layer_output={layer_output}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CompoundProteinInteractionPrediction(n_fingerprint=max(1, n_fingerprint+1),
                                             n_word=max(1, n_word+1),
                                             dim=dim,
                                             layer_gnn=layer_gnn,
                                             window=window,
                                             layer_cnn=layer_cnn,
                                             layer_output=layer_output).to(device)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found: {model_path}")

# load model state dict (best-effort)
state = torch.load(MODEL_PATH, map_location=device)
try:
    model.load_state_dict(state)
    print("Loaded model state_dict successfully.")
except Exception as e:
    # Possibly model file is a dict with additional keys; try 'model' or nested
    loaded = False
    if isinstance(state, dict):
        for key in ['model_state_dict', 'state_dict', 'model']:
            if key in state:
                try:
                    model.load_state_dict(state[key])
                    print(f"Loaded model from state['{key}'].")
                    loaded = True
                    break
                except Exception:
                    pass
    if not loaded:
        print("Failed to load state dict into model. Error:", e)
        print("Try saving the state_dict during training using torch.save(model.state_dict(), path).")
        sys.exit(1)

# ---------------------------
# Preprocess the single sample
# ---------------------------
mol = Chem.AddHs(Chem.MolFromSmiles(SMILES))
if mol is None:
    raise ValueError("RDKit failed to parse SMILES: " + SMILES)

atoms_tokens = create_atoms_for_prediction(mol)
i_jbond = create_ijbonddict(mol)
fingerprints = extract_fingerprints_for_prediction(atoms_tokens, i_jbond, radius)
adjacency = create_adjacency_for_prediction(mol)
words = split_sequence_for_prediction(SEQUENCE, ngram)

# Convert to tensors shaped like training tensors:
# In training, load_tensor returned a list of tensors where each entry was e.g. LongTensor([...]) and adjacency FloatTensor([...])
# For GNN: the model expects fingerprint tensors shaped (num_nodes,) -> embedding -> (num_nodes, dim)
fingerprints_t = torch.LongTensor(fingerprints).to(device)
adjacency_t = torch.FloatTensor(adjacency).to(device)
words_t = torch.LongTensor(words).to(device)

# The model forward expects inputs = (fingerprints, adjacency, words)
label, score, probs = model.predict((fingerprints_t, adjacency_t, words_t))

print("RESULT")
print("SMILES:", SMILES)
print("Protein sequence:", SEQUENCE)
print("Predicted class label:", label)
print("Predicted probability for class 1:", score)
print("All probabilities:", probs)
print("\nNote: If many fingerprint tokens were unknown (mapped to 0), prediction could be unreliable.")
print("If you see many unknown fallbacks, please re-run preprocessing so you save atom_dict/bond_dict/edge_dict too,")
print("or preprocess your query molecule together with training set vocabulary so keys match exactly.")


Loaded fingerprint_dict (size=3253) and word_dict (size=2098) from ../dataset/b_cancer/input/radius2_ngram3/
Using hyperparams dim=10, layer_gnn=3, window=11, layer_cnn=3, layer_output=3
Failed to load state dict into model. Error: Error(s) in loading state_dict for CompoundProteinInteractionPrediction:
	size mismatch for embed_fingerprint.weight: copying a param with shape torch.Size([3253, 10]) from checkpoint, the shape in current model is torch.Size([3254, 10]).
	size mismatch for embed_word.weight: copying a param with shape torch.Size([2098, 10]) from checkpoint, the shape in current model is torch.Size([2099, 10]).
Try saving the state_dict during training using torch.save(model.state_dict(), path).


SystemExit: 1

c:\Users\admin\anaconda3\envs\cpi\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


Try 3

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import defaultdict
import os
import pickle
from rdkit import Chem

# Sample inputs (REPLACE THESE with your actual SMILES and sequence)
SMILES = 'CCO'  # Example: Ethanol
SEQUENCE = 'MSEQ'  # Short example protein sequence; use a real one like 'MAVEEGVR...' for testing

# Simple Model Architecture (same as before)
class SimpleGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super(SimpleGNN, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
    
    def forward(self, x, adj):
        for layer in self.layers:
            x = torch.mm(adj, x)
            x = F.relu(layer(x))
        return x.mean(dim=0)  # Global mean pool

class SimpleCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, window_size):
        super(SimpleCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(nn.Conv1d(embed_dim, hidden_dim, kernel_size=window_size, padding=window_size//2))
    
    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)  # (batch, embed, seq)
        for conv in self.convs:
            x = F.relu(conv(x))
        return x.mean(dim=2).mean(dim=0)  # Global avg pool

class InteractionModel(nn.Module):
    def __init__(self, fingerprint_dim, word_vocab_size, dim, layer_gnn, layer_cnn, layer_output, window):
        super(InteractionModel, self).__init__()
        self.gnn = SimpleGNN(fingerprint_dim, dim, layer_gnn)
        self.cnn = SimpleCNN(word_vocab_size, dim, dim, layer_cnn, window)
        self.mlp = nn.ModuleList()
        self.mlp.append(nn.Linear(2 * dim, dim))
        for _ in range(layer_output - 1):
            self.mlp.append(nn.Linear(dim, dim))
        self.fc = nn.Linear(dim, 1)
    
    def forward(self, fingerprints, adjacency, words):
        compound_emb = self.gnn(fingerprints, adjacency)
        protein_emb = self.cnn(words)
        combined = torch.cat([compound_emb, protein_emb], dim=0)
        for layer in self.mlp:
            combined = F.relu(layer(combined))
        out = torch.sigmoid(self.fc(combined))
        return out

# Preprocessing functions (unchanged)
def create_atoms(mol, atom_dict):
    atoms = [a.GetSymbol() for a in mol.GetAtoms()]
    for a in mol.GetAromaticAtoms():
        i = a.GetIdx()
        atoms[i] = (atoms[i], 'aromatic')
    atoms = [atom_dict[a] for a in atoms]
    return np.array(atoms)

def create_ijbonddict(mol, bond_dict):
    i_jbond_dict = defaultdict(lambda: [])
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bond = bond_dict[str(b.GetBondType())]
        i_jbond_dict[i].append((j, bond))
        i_jbond_dict[j].append((i, bond))
    return i_jbond_dict

def extract_fingerprints(atoms, i_jbond_dict, radius, fingerprint_dict, edge_dict):
    if (len(atoms) == 1) or (radius == 0):
        fingerprints = [fingerprint_dict[a] for a in atoms]
    else:
        nodes = atoms
        i_jedge_dict = i_jbond_dict
        for _ in range(radius):
            fingerprints = []
            for i, j_edge in i_jedge_dict.items():
                neighbors = [(nodes[j], edge) for j, edge in j_edge]
                fingerprint = (nodes[i], tuple(sorted(neighbors)))
                fingerprints.append(fingerprint_dict[fingerprint])
            nodes = fingerprints
            _i_jedge_dict = defaultdict(lambda: [])
            for i, j_edge in i_jedge_dict.items():
                for j, edge in j_edge:
                    both_side = tuple(sorted((nodes[i], nodes[j])))
                    edge = edge_dict[(both_side, edge)]
                    _i_jedge_dict[i].append((j, edge))
            i_jedge_dict = _i_jedge_dict
    return np.array(fingerprints)

def create_adjacency(mol):
    adjacency = Chem.GetAdjacencyMatrix(mol)
    return np.array(adjacency)

def split_sequence(sequence, ngram, word_dict):
    sequence = '-' + sequence + '='
    words = [word_dict[sequence[i:i+ngram]] for i in range(len(sequence)-ngram+1)]
    return np.array(words)

def load_dictionaries(input_dir):
    with open(os.path.join(input_dir, 'atom_dict.pickle'), 'rb') as f:
        atom_dict = pickle.load(f)
    with open(os.path.join(input_dir, 'bond_dict.pickle'), 'rb') as f:
        bond_dict = pickle.load(f)
    with open(os.path.join(input_dir, 'fingerprint_dict.pickle'), 'rb') as f:
        fingerprint_dict = pickle.load(f)
    with open(os.path.join(input_dir, 'edge_dict.pickle'), 'rb') as f:
        edge_dict = pickle.load(f)
    with open(os.path.join(input_dir, 'word_dict.pickle'), 'rb') as f:
        word_dict = pickle.load(f)
    return atom_dict, bond_dict, fingerprint_dict, edge_dict, word_dict

def preprocess_single_sample(smiles, sequence, atom_dict, bond_dict, fingerprint_dict, edge_dict, word_dict, radius, ngram):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    atoms = create_atoms(mol, atom_dict)
    i_jbond_dict = create_ijbonddict(mol, bond_dict)
    fingerprints = extract_fingerprints(atoms, i_jbond_dict, radius, fingerprint_dict, edge_dict)
    adjacency = create_adjacency(mol)
    words = split_sequence(sequence, ngram, word_dict)
    return fingerprints, adjacency, words

# Main execution (hardcoded params from batch script)
DATASET = 'b_cancer'
radius = 2
ngram = 3
dim = 10
layer_gnn = 3
side = 5
window = 2 * side + 1
layer_cnn = 3
layer_output = 3
setting = f"{DATASET}--radius{radius}--ngram{ngram}--dim{dim}--layer_gnn{layer_gnn}--window{window}--layer_cnn{layer_cnn}--layer_output{layer_output}--lr1e-3--lr_decay0.5--decay_interval10--weight_decay1e-6--iteration100"

# Paths (adjust if your model save path differs)
input_dir = os.path.join('../dataset', DATASET, 'input', f'radius{radius}_ngram{ngram}')
model_path = os.path.join('../models', DATASET, setting, 'model.pth')

# Load dicts
atom_dict, bond_dict, fingerprint_dict, edge_dict, word_dict = load_dictionaries(input_dir)
vocab_size = len(word_dict)
fingerprint_dim = max(fingerprint_dict.values()) + 1 if isinstance(list(fingerprint_dict.values())[0], int) else len(fingerprint_dict)

# Preprocess
fingerprints, adjacency, words = preprocess_single_sample(
    SMILES, SEQUENCE, atom_dict, bond_dict, fingerprint_dict, edge_dict, word_dict, radius, ngram
)

# To tensors (batch size 1)
fingerprints_t = torch.tensor(fingerprints, dtype=torch.float).unsqueeze(0)
adjacency_t = torch.tensor(adjacency, dtype=torch.float).unsqueeze(0)
words_t = torch.tensor(words, dtype=torch.long).unsqueeze(0)

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = InteractionModel(fingerprint_dim, vocab_size, dim, layer_gnn, layer_cnn, layer_output, window).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

with torch.no_grad():
    pred_prob = model(fingerprints_t.to(device), adjacency_t.to(device), words_t.to(device)).item()
    pred_binary = 1 if pred_prob > 0.5 else 0

print(f"Input SMILES: {SMILES}")
print(f"Input Sequence: {SEQUENCE}")
print(f"Predicted probability: {pred_prob:.4f}")
print(f"Predicted interaction (0/1): {pred_binary}")

FileNotFoundError: [Errno 2] No such file or directory: '../dataset\\b_cancer\\input\\radius2_ngram3\\atom_dict.pickle'

In [30]:
# Try 4

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import defaultdict
import os
import pickle
from rdkit import Chem

# Sample inputs (REPLACE THESE with your actual SMILES and sequence)
SMILES = 'CCO'  # Example: Ethanol
SEQUENCE = 'MSEQ'  # Short example protein sequence; use a real one

# Simple Model Architecture (placeholder - replace with your actual model from run_training.py if different)
class SimpleGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super(SimpleGNN, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
    
    def forward(self, x, adj):
        for layer in self.layers:
            x = torch.mm(adj, x)
            x = F.relu(layer(x))
        return x.mean(dim=0)  # Global mean pool

class SimpleCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, window_size):
        super(SimpleCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(nn.Conv1d(embed_dim, hidden_dim, kernel_size=window_size, padding=window_size//2))
    
    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)  # (batch, embed, seq)
        for conv in self.convs:
            x = F.relu(conv(x))
        return x.mean(dim=2).mean(dim=0)  # Global avg pool

class InteractionModel(nn.Module):
    def __init__(self, fingerprint_dim, word_vocab_size, dim, layer_gnn, layer_cnn, layer_output, window):
        super(InteractionModel, self).__init__()
        self.gnn = SimpleGNN(fingerprint_dim, dim, layer_gnn)
        self.cnn = SimpleCNN(word_vocab_size, dim, dim, layer_cnn, window)
        self.mlp = nn.ModuleList()
        self.mlp.append(nn.Linear(2 * dim, dim))
        for _ in range(layer_output - 1):
            self.mlp.append(nn.Linear(dim, dim))
        self.fc = nn.Linear(dim, 1)
    
    def forward(self, fingerprints, adjacency, words):
        compound_emb = self.gnn(fingerprints, adjacency)
        protein_emb = self.cnn(words)
        combined = torch.cat([compound_emb, protein_emb], dim=0)
        for layer in self.mlp:
            combined = F.relu(layer(combined))
        out = torch.sigmoid(self.fc(combined))
        return out

# Preprocessing functions
def create_atoms(mol, atom_dict):
    atoms = [a.GetSymbol() for a in mol.GetAtoms()]
    for a in mol.GetAromaticAtoms():
        i = a.GetIdx()
        atoms[i] = (atoms[i], 'aromatic')
    atoms = [atom_dict[a] for a in atoms]
    return np.array(atoms)

def create_ijbonddict(mol, bond_dict):
    i_jbond_dict = defaultdict(lambda: [])
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bond = bond_dict[str(b.GetBondType())]
        i_jbond_dict[i].append((j, bond))
        i_jbond_dict[j].append((i, bond))
    return i_jbond_dict

def extract_fingerprints(atoms, i_jbond_dict, radius, fingerprint_dict, edge_dict):
    if (len(atoms) == 1) or (radius == 0):
        fingerprints = [fingerprint_dict[a] for a in atoms]
    else:
        nodes = atoms
        i_jedge_dict = i_jbond_dict
        for _ in range(radius):
            fingerprints = []
            for i, j_edge in i_jedge_dict.items():
                neighbors = [(nodes[j], edge) for j, edge in j_edge]
                fingerprint = (nodes[i], tuple(sorted(neighbors)))
                fingerprints.append(fingerprint_dict[fingerprint])
            nodes = fingerprints
            _i_jedge_dict = defaultdict(lambda: [])
            for i, j_edge in i_jedge_dict.items():
                for j, edge in j_edge:
                    both_side = tuple(sorted((nodes[i], nodes[j])))
                    edge = edge_dict[(both_side, edge)]
                    _i_jedge_dict[i].append((j, edge))
            i_jedge_dict = _i_jedge_dict
    return np.array(fingerprints)

def create_adjacency(mol):
    adjacency = Chem.GetAdjacencyMatrix(mol)
    return np.array(adjacency)

def split_sequence(sequence, ngram, word_dict):
    sequence = '-' + sequence + '='
    words = [word_dict[sequence[i:i+ngram]] for i in range(len(sequence)-ngram+1)]
    return np.array(words)

def build_all_dicts(dataset_dir, radius, ngram):
    """Build all required dictionaries by processing the original training data."""
    original_file = os.path.join(dataset_dir, 'original', 'data.txt')
    if not os.path.exists(original_file):
        raise FileNotFoundError(f"Original data file not found: {original_file}")
    
    with open(original_file, 'r') as f:
        data_list = f.read().strip().split('\n')
    
    data_list = [d for d in data_list if '.' not in d.strip().split()[0]]
    
    atom_dict = defaultdict(lambda: len(atom_dict))
    bond_dict = defaultdict(lambda: len(bond_dict))
    fingerprint_dict = defaultdict(lambda: len(fingerprint_dict))
    edge_dict = defaultdict(lambda: len(edge_dict))
    word_dict = defaultdict(lambda: len(word_dict))
    
    for data in data_list:
        smiles, sequence, _ = data.strip().split()
        
        # Compound dicts
        mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
        atoms = [a.GetSymbol() for a in mol.GetAtoms()]
        for a in mol.GetAromaticAtoms():
            i = a.GetIdx()
            atoms[i] = (atoms[i], 'aromatic')
        atoms_ids = [atom_dict[a] for a in atoms]
        
        i_jbond_dict = defaultdict(lambda: [])
        for b in mol.GetBonds():
            i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
            bond = str(b.GetBondType())
            bond_id = bond_dict[bond]
            i_jbond_dict[i].append((j, bond_id))
            i_jbond_dict[j].append((i, bond_id))
        
        _ = extract_fingerprints(atoms_ids, i_jbond_dict, radius, fingerprint_dict, edge_dict)
        
        # Protein dict
        seq_padded = '-' + sequence + '='
        for i in range(len(seq_padded) - ngram + 1):
            ngram_str = seq_padded[i:i + ngram]
            _ = word_dict[ngram_str]
    
    # Convert to regular dicts for pickling/loading if needed
    return dict(atom_dict), dict(bond_dict), dict(fingerprint_dict), dict(edge_dict), dict(word_dict)

def preprocess_single_sample(smiles, sequence, atom_dict, bond_dict, fingerprint_dict, edge_dict, word_dict, radius, ngram):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    atoms = create_atoms(mol, atom_dict)
    i_jbond_dict = create_ijbonddict(mol, bond_dict)
    fingerprints = extract_fingerprints(atoms, i_jbond_dict, radius, fingerprint_dict, edge_dict)
    adjacency = create_adjacency(mol)
    words = split_sequence(sequence, ngram, word_dict)
    return fingerprints, adjacency, words

# Main execution (hardcoded params from batch script)
DATASET = 'b_cancer'
radius = 2
ngram = 3
dim = 10
layer_gnn = 3
side = 5
window = 2 * side + 1
layer_cnn = 3
layer_output = 3
setting = f"{DATASET}--radius{radius}--ngram{ngram}--dim{dim}--layer_gnn{layer_gnn}--window{window}--layer_cnn{layer_cnn}--layer_output{layer_output}--lr1e-3--lr_decay0.5--decay_interval10--weight_decay1e-6--iteration100"

# Paths (adjust model path if your training saves differently)
dataset_dir = '../dataset/' + DATASET
input_dir = os.path.join(dataset_dir, 'input', f'radius{radius}_ngram{ngram}')
model_path = os.path.join('../models', DATASET, setting, 'model.pth')  # e.g., from training output

# Build dictionaries from original data
atom_dict, bond_dict, fingerprint_dict, edge_dict, word_dict = build_all_dicts(dataset_dir, radius, ngram)

vocab_size = len(word_dict)
fingerprint_dim = len(fingerprint_dict)

# Preprocess sample
fingerprints, adjacency, words = preprocess_single_sample(
    SMILES, SEQUENCE, atom_dict, bond_dict, fingerprint_dict, edge_dict, word_dict, radius, ngram
)

# To tensors (batch size 1; adjust if model expects specific padding)
fingerprints_t = torch.tensor(fingerprints[np.newaxis, :], dtype=torch.float)  # (1, num_atoms)
adjacency_t = torch.tensor(adjacency[np.newaxis, :, :], dtype=torch.float)  # (1, num_atoms, num_atoms)
words_t = torch.tensor(words[np.newaxis, :], dtype=torch.long)  # (1, seq_len)

# Load model (placeholder architecture - replace with actual if needed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = InteractionModel(fingerprint_dim, vocab_size, dim, layer_gnn, layer_cnn, layer_output, window).to(device)

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
else:
    print(f"Warning: Model file not found at {model_path}. Using untrained model (random predictions).")
model.eval()

with torch.no_grad():
    pred_prob = model(fingerprints_t.to(device), adjacency_t.to(device), words_t.to(device)).item()
    pred_binary = 1 if pred_prob > 0.5 else 0

print(f"Input SMILES: {SMILES}")
print(f"Input Sequence: {SEQUENCE}")
print(f"Predicted probability: {pred_prob:.4f}")
print(f"Predicted interaction (0/1): {pred_binary}")

KeyError: (78, ((12, 114), (12, 114), (69, 2168), (81, 1474)))